# Aprendizaje de Simbolo: `ParabolicMLPSymbol` con datos GRF (laplaciano fraccionario 2D)

Mismo experimento que `03_barrido_grf.ipynb` con la referencia $a(\mathbf{k})=|\mathbf{k}|^\alpha$
(`FractionalLaplacianSymbol`, $n=2$) para $\alpha\in\{0.5,\ 1,\ 1.5\}$, aprendiendo con orden $m=\alpha$
($\rho=n+1=3$ para el simbolo aprendido). Hay una seccion por $\alpha$ y, al final, una tabla resumen.

**Funcion de perdida** (la misma de 03)

$$\mathcal{L} = \lambda_{\text{traj}}\,\mathcal{L}_{\text{traj}} + \lambda_{\text{par}}\,\mathcal{L}_{\text{par}} + \lambda_{\text{class}}\,\mathcal{L}_{\text{class}}$$

- **Trayectoria:** $\mathcal{L}_{\text{traj}} = \mathbb{E}\big[|e^{-t a_\theta(\mathbf{k})}\hat u_0(\mathbf{k}) - \hat u_{\text{ref}}(\mathbf{k},t)|^2\big]$
- **Parabolicidad (Def. 2.3):** $\mathcal{L}_{\text{par}} = \mathbb{E}\big[\max\big(0,\ |(a_\theta(\mathbf{k})+\lambda)^{-1}| - \kappa(1+|(\mathbf{k},\lambda)|^2)^{-m/2}\big)^2\big]$,
  sobre $\mathrm{Re}(\lambda)\ge 0$, $|(\mathbf{k},\lambda)|\ge\omega$.
- **Clase de simbolo $S^{m,\rho}$:**
  $\mathcal{L}_{\text{class}} = \mathbb{E}_{\mathbf{k}}\Big[\frac{1}{|A_\rho|}\sum_{\alpha\in A_\rho}\max\big(0,\ \langle\mathbf{k}\rangle^{|\alpha|-m}\,|\Delta^\alpha_k a_\theta(\mathbf{k})| - C_\alpha\big)^2\Big]$,
  con $A_\rho=\{\alpha\in\mathbb{N}_0^n : |\alpha|\le\rho\}$ y $\langle\mathbf{k}\rangle=(1+|\mathbf{k}|^2)^{1/2}$.

# Setup
Configuracion identica a `03_barrido_grf.ipynb`; el codigo de cada seccion esta en
`parabolic_symbol.learning.experiment.GRFExperiment`.

- **Datos (`data/grf.py`):** toro $\mathbb{T}^2=[0,2\pi)^2$, $u(x)=\sum_k \hat u(k)e^{ik\cdot x}$,
  $u_0\sim\mathcal N\big(0,\ 7^4(-\Delta+7^2)^{-2.5}\big)$ en una malla de $128\times128$ puntos.
  Entrenamiento: $100$ condiciones iniciales, $t=0.05+0.1j$, $j=0,\dots,9$. Prueba: $20$ condiciones
  iniciales nuevas, $t=0.05+0.05j$, $j=0,\dots,19$.
- **Mascara de recuperabilidad:** se descartan los pares $(k,t)$ con $e^{-t\,a(k)}\le\varepsilon_{\texttt{float64}}$,
  y los $k$ con $|k|_\infty > k_{\text{cap}} = 32$. La ventana de $k$ que queda fija $k_{\max}$.
  Particion entrenamiento / validacion **por $\mathbf{k}$** (80% / 20% de los $k$ unicos).
- **$\kappa$:** $\kappa=$ `theoretical_kappa()`, con $c$ y $C$ estimados de los datos de entrenamiento
  (`estimate_symbol` + `estimate_bounds`, $\delta=0.1$) con $m=\alpha$. Las $C_\alpha$ de $\mathcal{L}_{\text{class}}$
  salen del simbolo analitico en $|k_i|\le k_{\max}$, con margen $1.5$.
- **Barrido:** $\{2,3,4\}$ capas ocultas $\times$ $\{32,64,128\}$ neuronas $\times$ dropout $0.2$ en `none` /
  `middle` $\times$ weight decay $\{0,\ 10^{-4}\}$, con **early stopping** sobre la perdida total de validacion
  (trayectoria + parabolicidad + $\mathcal{L}_{\text{class}}$). Se elige la corrida de menor error relativo de validacion.
- `cuda` si esta disponible y `float64`.

In [ ]:
from pathlib import Path

import torch

from parabolic_symbol.symbols.heat import HeatSymbol
from parabolic_symbol.symbols.fractional_laplacian import FractionalLaplacianSymbol
from parabolic_symbol.learning.experiment import GRFExperiment, summary_table
from parabolic_symbol.utils import default_device

device = default_device()
dtype = torch.float64
print(f"device = {device}  |  dtype = {dtype}")

RESULTS = Path("../results/06_fraccionario_2d")   # tablas (CSV) y figuras (PNG)
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
ALPHAS = [0.5, 1.0, 1.5]
experimentos = {
    alpha: GRFExperiment(
        reference=FractionalLaplacianSymbol(n=2, alpha=alpha), m=alpha, resolution=128, k_cap=32,
        results=RESULTS / f"alpha_{alpha:g}", dtype=dtype, device=device,
    )
    for alpha in ALPHAS
}

# $\alpha = 0.5$

$a(\mathbf{k})=|\mathbf{k}|^{0.5}$, $m=0.5$, malla 128x128, $k_{\text{cap}}=32$. Tablas y figuras en `results/06_fraccionario_2d/alpha_0.5/`.

In [ ]:
exp_0_5 = experimentos[0.5]

## 1. Datos de referencia

Ventana de $k$ resultante y numero de muestras por conjunto.

In [ ]:
display(exp_0_5.load_data())

## 2. $\kappa$ teorico y constantes $C_\alpha$

$c$ y $C$ se estiman solo con los datos de entrenamiento y fijan $\kappa=$ `theoretical_kappa()`. $\hat\kappa$ del simbolo de referencia se calcula solo como comparacion.

In [ ]:
display(exp_0_5.estimate_constants())

## 3. Barrido de hiperparametros

$3$ profundidades $\times$ $3$ anchos $\times$ ubicacion del dropout $\times$ $2$ weight decays $=$ **30 entrenamientos** (con $2$ capas, `middle` equivale a `none` y no se repite).

In [ ]:
exp_0_5.sweep()

## 4. Resultados del barrido

In [ ]:
barrido, por_eje = exp_0_5.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

Todas las corridas usan $\kappa=$ `theoretical_kappa()`, asi que se elige la de **menor error relativo de validacion**.

In [ ]:
display(exp_0_5.select())

### Metricas del modelo seleccionado

Trayectoria sobre las muestras de entrenamiento y validacion, y simbolo $a_\theta(\mathbf{{k}})$ frente a $|\mathbf{k}|^{0.5}$ sobre los $k$ de entrenamiento y los de validacion (que la red nunca vio).

In [ ]:
traj, sym = exp_0_5.metrics()
display(traj)
display(sym)

### Evaluacion en prueba

Error relativo $L^2$ sobre los pares $(k,t)$ recuperables de prueba y en espacio fisico (malla completa, todos los modos), por condicion inicial y tiempo.

In [ ]:
display(exp_0_5.test())

## 5. Curvas de entrenamiento

Perdida total de `evaluate_split` (incluye $\mathcal{{L}}_{{\text{{class}}}}$) y error relativo; la linea punteada marca la epoca que restauro el early stopping.

In [ ]:
exp_0_5.curves()

## 6. Simbolo aprendido vs. referencia

Simbolo frente a $|\mathbf{k}|^{0.5}$ en $|k_i|\le k_{\max}$: curva contra $|k|$, paridad y error absoluto contra $|k|$ (el simbolo es radial), y un mapa del error relativo sobre $(k_1,k_2)$.

In [ ]:
exp_0_5.symbol_figure()

## 7. Certificacion de parabolicidad

Rejilla independiente de la de entrenamiento: $|\lambda|$ log-espaciado en $[10^{-2},10^3]$ con $25$ angulos en $[-\pi/2,\pi/2]$, y $|k_i|\le K$ con $K=k_{\max}$ (dentro de la ventana), $2k_{\max}$ y $3k_{\max}$ (fuera).

In [ ]:
display(exp_0_5.certification())

## 8. Clase del simbolo: $S^{m,\rho}$

`SymbolClassVerifier` estima $\hat C_\alpha(K)$ en ventanas crecientes y la pendiente log-log. **Es evidencia en una ventana finita, no una prueba.**

In [ ]:
display(exp_0_5.symbol_class())

# $\alpha = 1$

$a(\mathbf{k})=|\mathbf{k}|^{1}$, $m=1$, malla 128x128, $k_{\text{cap}}=32$. Tablas y figuras en `results/06_fraccionario_2d/alpha_1/`.

In [ ]:
exp_1_0 = experimentos[1.0]

## 1. Datos de referencia

Ventana de $k$ resultante y numero de muestras por conjunto.

In [ ]:
display(exp_1_0.load_data())

## 2. $\kappa$ teorico y constantes $C_\alpha$

$c$ y $C$ se estiman solo con los datos de entrenamiento y fijan $\kappa=$ `theoretical_kappa()`. $\hat\kappa$ del simbolo de referencia se calcula solo como comparacion.

In [ ]:
display(exp_1_0.estimate_constants())

## 3. Barrido de hiperparametros

$3$ profundidades $\times$ $3$ anchos $\times$ ubicacion del dropout $\times$ $2$ weight decays $=$ **30 entrenamientos** (con $2$ capas, `middle` equivale a `none` y no se repite).

In [ ]:
exp_1_0.sweep()

## 4. Resultados del barrido

In [ ]:
barrido, por_eje = exp_1_0.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

Todas las corridas usan $\kappa=$ `theoretical_kappa()`, asi que se elige la de **menor error relativo de validacion**.

In [ ]:
display(exp_1_0.select())

### Metricas del modelo seleccionado

Trayectoria sobre las muestras de entrenamiento y validacion, y simbolo $a_\theta(\mathbf{{k}})$ frente a $|\mathbf{k}|^{1}$ sobre los $k$ de entrenamiento y los de validacion (que la red nunca vio).

In [ ]:
traj, sym = exp_1_0.metrics()
display(traj)
display(sym)

### Evaluacion en prueba

Error relativo $L^2$ sobre los pares $(k,t)$ recuperables de prueba y en espacio fisico (malla completa, todos los modos), por condicion inicial y tiempo.

In [ ]:
display(exp_1_0.test())

## 5. Curvas de entrenamiento

Perdida total de `evaluate_split` (incluye $\mathcal{{L}}_{{\text{{class}}}}$) y error relativo; la linea punteada marca la epoca que restauro el early stopping.

In [ ]:
exp_1_0.curves()

## 6. Simbolo aprendido vs. referencia

Simbolo frente a $|\mathbf{k}|^{1}$ en $|k_i|\le k_{\max}$: curva contra $|k|$, paridad y error absoluto contra $|k|$ (el simbolo es radial), y un mapa del error relativo sobre $(k_1,k_2)$.

In [ ]:
exp_1_0.symbol_figure()

## 7. Certificacion de parabolicidad

Rejilla independiente de la de entrenamiento: $|\lambda|$ log-espaciado en $[10^{-2},10^3]$ con $25$ angulos en $[-\pi/2,\pi/2]$, y $|k_i|\le K$ con $K=k_{\max}$ (dentro de la ventana), $2k_{\max}$ y $3k_{\max}$ (fuera).

In [ ]:
display(exp_1_0.certification())

## 8. Clase del simbolo: $S^{m,\rho}$

`SymbolClassVerifier` estima $\hat C_\alpha(K)$ en ventanas crecientes y la pendiente log-log. **Es evidencia en una ventana finita, no una prueba.**

In [ ]:
display(exp_1_0.symbol_class())

# $\alpha = 1.5$

$a(\mathbf{k})=|\mathbf{k}|^{1.5}$, $m=1.5$, malla 128x128, $k_{\text{cap}}=32$. Tablas y figuras en `results/06_fraccionario_2d/alpha_1.5/`.

In [ ]:
exp_1_5 = experimentos[1.5]

## 1. Datos de referencia

Ventana de $k$ resultante y numero de muestras por conjunto.

In [ ]:
display(exp_1_5.load_data())

## 2. $\kappa$ teorico y constantes $C_\alpha$

$c$ y $C$ se estiman solo con los datos de entrenamiento y fijan $\kappa=$ `theoretical_kappa()`. $\hat\kappa$ del simbolo de referencia se calcula solo como comparacion.

In [ ]:
display(exp_1_5.estimate_constants())

## 3. Barrido de hiperparametros

$3$ profundidades $\times$ $3$ anchos $\times$ ubicacion del dropout $\times$ $2$ weight decays $=$ **30 entrenamientos** (con $2$ capas, `middle` equivale a `none` y no se repite).

In [ ]:
exp_1_5.sweep()

## 4. Resultados del barrido

In [ ]:
barrido, por_eje = exp_1_5.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

Todas las corridas usan $\kappa=$ `theoretical_kappa()`, asi que se elige la de **menor error relativo de validacion**.

In [ ]:
display(exp_1_5.select())

### Metricas del modelo seleccionado

Trayectoria sobre las muestras de entrenamiento y validacion, y simbolo $a_\theta(\mathbf{{k}})$ frente a $|\mathbf{k}|^{1.5}$ sobre los $k$ de entrenamiento y los de validacion (que la red nunca vio).

In [ ]:
traj, sym = exp_1_5.metrics()
display(traj)
display(sym)

### Evaluacion en prueba

Error relativo $L^2$ sobre los pares $(k,t)$ recuperables de prueba y en espacio fisico (malla completa, todos los modos), por condicion inicial y tiempo.

In [ ]:
display(exp_1_5.test())

## 5. Curvas de entrenamiento

Perdida total de `evaluate_split` (incluye $\mathcal{{L}}_{{\text{{class}}}}$) y error relativo; la linea punteada marca la epoca que restauro el early stopping.

In [ ]:
exp_1_5.curves()

## 6. Simbolo aprendido vs. referencia

Simbolo frente a $|\mathbf{k}|^{1.5}$ en $|k_i|\le k_{\max}$: curva contra $|k|$, paridad y error absoluto contra $|k|$ (el simbolo es radial), y un mapa del error relativo sobre $(k_1,k_2)$.

In [ ]:
exp_1_5.symbol_figure()

## 7. Certificacion de parabolicidad

Rejilla independiente de la de entrenamiento: $|\lambda|$ log-espaciado en $[10^{-2},10^3]$ con $25$ angulos en $[-\pi/2,\pi/2]$, y $|k_i|\le K$ con $K=k_{\max}$ (dentro de la ventana), $2k_{\max}$ y $3k_{\max}$ (fuera).

In [ ]:
display(exp_1_5.certification())

## 8. Clase del simbolo: $S^{m,\rho}$

`SymbolClassVerifier` estima $\hat C_\alpha(K)$ en ventanas crecientes y la pendiente log-log. **Es evidencia en una ventana finita, no una prueba.**

In [ ]:
display(exp_1_5.symbol_class())

# Resumen

Una fila por $\alpha$: ventana de $k$, $c$, $C$, $\kappa$, configuracion elegida, error relativo de validacion, error en prueba (pares recuperables y espacio fisico), error relativo medio del simbolo en $|k_i|\le k_{\max}$, peor ratio de la certificacion (en las tres ventanas) y multiindices consistentes. Se guarda en `resumen.csv`.

In [ ]:
display(summary_table(experimentos, RESULTS))